# 01 · Exploring the German Credit Data

**Goal of this notebook:** understand the dataset before analysing it: its size, its columns, its data types, missing values and the target.

**Data:** 1,000 loan applicants from a German bank (UCI Statlog German Credit). `class`: 1 = good, 2 = bad. Code meanings: see `DATA_DICTIONARY.md`.

In [1]:
import numpy as np
import pandas as pd

# Load the data into a table (DataFrame) called df
df = pd.read_csv("data/german_credit.csv")

## 1. First look at the data

In [2]:
# Show the first 5 rows (5 is the default)
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings,employment,installment_rate,personal_status,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,class
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


**What I see:** Each row is one loan customer and each column is one piece of information about them. Some columns are numbers (duration, credit_amount, age), but many are codes like `A11` or `A34`, which stand for categories (see `DATA_DICTIONARY.md`).

## 2. Size of the dataset (rows, columns)

In [3]:
# shape is an attribute (a stored fact), so no brackets
df.shape

(1000, 21)

**What I see:** The dataset has **1,000 rows** (one per loan customer) and **21 columns**: 20 pieces of information about each customer, plus `class`, which says whether they were a good or bad customer.

## 3. Column names and data types

In [4]:
# The type of values in every column
df.dtypes

checking_status            object
duration                    int64
credit_history             object
purpose                    object
credit_amount               int64
savings                    object
employment                 object
installment_rate            int64
personal_status            object
other_debtors              object
residence_since             int64
property                   object
age                         int64
other_installment_plans    object
housing                    object
existing_credits            int64
job                        object
num_dependents              int64
telephone                  object
foreign_worker             object
class                       int64
dtype: object

In [5]:
# How many columns of each type
df.dtypes.value_counts()

object    13
int64      8
Name: count, dtype: int64

**What I see:** **8 columns are numbers** (`int64`: e.g. age, loan amount) and **13 columns are text codes** (`object`: e.g. `A11`) that stand for categories. These codes need to be translated into readable words before analysis.

## 4. Missing values

In [6]:
# isnull() marks every empty cell as True; sum() counts the Trues per column
df.isnull().sum()

checking_status            0
duration                   0
credit_history             0
purpose                    0
credit_amount              0
savings                    0
employment                 0
installment_rate           0
personal_status            0
other_debtors              0
residence_since            0
property                   0
age                        0
other_installment_plans    0
housing                    0
existing_credits           0
job                        0
num_dependents             0
telephone                  0
foreign_worker             0
class                      0
dtype: int64

**What I see:** There are **no missing values** in any column, so nothing needs to be filled in or removed.

## 5. The target: how many good vs bad customers?

In [7]:
# Count how often each value appears in the class column
df["class"].value_counts()

class
1    700
2    300
Name: count, dtype: int64

In [8]:
# The same as percentages
df["class"].value_counts(normalize=True)

class
1    0.7
2    0.3
Name: proportion, dtype: float64

**What I see:** **700 customers (70%) paid back** their loan and **300 (30%) did not**. The data is **imbalanced**. A model that always guesses "good" would already be 70% correct, so this is the **baseline** any real model must beat.

## 6. Summary statistics of the numeric columns

In [9]:
# count, mean, std, min, quartiles (50% = median) and max of every numeric column
df.describe()

,duration,credit_amount,installment_rate,residence_since,age,existing_credits,num_dependents,class
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,1.300000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,2.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,2.000000


**What I see:** The average loan is **3,271 DM** for about **21 months**, and the average customer is **35 years old**. The average loan amount (3,271) is much higher than the median (2,320), so a few very large loans (up to 18,424 DM) pull the average up. Most loans are smaller than the average suggests.

## 7. A closer look at age with NumPy

In [10]:
# Take the age column out of pandas as a plain NumPy array
ages = df["age"].to_numpy()

print("Average:", ages.mean())
print("Median: ", np.median(ages))
print("Std:    ", ages.std())
print("Youngest:", ages.min(), "| Oldest:", ages.max())

# ages < 25 gives True/False per customer; the mean of True/False = share of True
print("Share under 25:", (ages < 25).mean())

Average: 35.546
Median:  33.0
Std:     11.36977941738537
Youngest: 19 | Oldest: 75
Share under 25: 0.149


**What I see:** Customers are between **19 and 75** years old, with an average of **35.5**. About **15% are younger than 25**. Later we'll check whether these young customers are riskier.

## Summary of this notebook
1. The data has **1,000 customers and 21 columns**, with **no missing values**.
2. **30% of customers are bad** (didn't pay back), so the data is imbalanced (baseline accuracy: 70%).
3. **13 columns are coded categories** (`A11`, ...) that must be translated before analysis. That's the next notebook, `02_clean.ipynb`.